# GeoAtt-PointNet++ — Evaluasi (Local)
Rahmat Zulfikri · Magister Teknik Elektro UGM

Notebook evaluasi lokal (tanpa Google Colab/Drive).  
Metrik lengkap: EER, AUC, TAR@FAR=1%, TAR@FAR=0.1%, d-prime, Accuracy@EER.  
Plot: ROC, DET, distribusi similarity, t-SNE, CMC, confusion matrix.

## 1. Setup & Import

In [ ]:
import sys
import os
from pathlib import Path

# Tambah root project ke sys.path agar semua module bisa di-import
PROJECT_ROOT = Path(".").resolve()  # sesuaikan jika notebook dijalankan dari folder lain
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 120

# Deteksi device (MPS untuk Apple Silicon, CUDA untuk NVIDIA, CPU fallback)
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")
print(f"Project : {PROJECT_ROOT}")

In [ ]:
# Import modules project
from models.siamese import SiamesePalmNet
from utils.dataset import (
    PalmPairDataset,
    scan_dataset,
    scan_dataset_frames,
    balance_label_sessions,
    balance_label_frames,
    generate_pairs,
    GEOMETRY_DIM,
)
from utils.metrics import (
    compute_all_metrics,
    compute_eer,
    compute_dprime,
    compute_tar_at_far,
    plot_roc,
    plot_det,
    plot_similarity_dist,
    plot_tsne,
    print_metrics_table,
)
from utils.normalizer import GeometryNormalizer
from evaluate import load_model, run_inference, evaluate_model, _is_frame_layout

print(f"GEOMETRY_DIM = {GEOMETRY_DIM}")
print("Import berhasil.")

## 2. Konfigurasi

In [ ]:
# ============================================================
# KONFIGURASI — sesuaikan bagian ini sebelum menjalankan
# ============================================================

# Root direktori dataset (frame layout: label/session/frame_xx/)
DATA_DIR = PROJECT_ROOT / "dataset"

# Checkpoint model yang ingin dievaluasi.
# Format: {"NamaModel": "path/ke/checkpoint.pth"}
# Contoh multi-model untuk ablation study:
CHECKPOINTS = {
    "M4_Fold0": "runs/local_exp1/fold_0/best.pth",
    # "M1_Fold0": "runs/m1/fold_0/best.pth",
    # "M4_Finetune": "runs/local_exp1/fold_0/best_finetuned.pth",
}

# Path normalizer.json (None = cari otomatis di sebelah checkpoint)
NORMALIZER_PATH = None

# Folder output untuk semua hasil evaluasi
OUTPUT_DIR = PROJECT_ROOT / "eval_results"

# Standarisasi dataset: cap setiap label ke jumlah sesi minimum
# True = fair evaluation; False = gunakan seluruh data yang tersedia
BALANCE_DATASET = True

# Hyperparameter inference
N_POINTS   = 4096     # titik yang di-sample per point cloud
SAMPLING   = "random" # "random" atau "fps"
BATCH_SIZE = 16
NUM_WORKERS = 0       # 0 = aman untuk macOS (MPS tidak support multi-worker)

# Simpan labels+scores ke .npz per model
SAVE_SCORES = False

# ============================================================
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR    : {DATA_DIR.resolve()}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR.resolve()}")
print(f"BALANCE     : {BALANCE_DATASET}")
print(f"Checkpoints : {list(CHECKPOINTS.keys())}")

## 3. Load Dataset

In [ ]:
# Auto-detect layout: frame atau session
frame_layout = _is_frame_layout(DATA_DIR)
print(f"Layout data : {'frame (dataset)' if frame_layout else 'session (result)'}")

if frame_layout:
    label_frames, session_groups = scan_dataset_frames(DATA_DIR)

    if BALANCE_DATASET:
        session_groups, min_sessions = balance_label_frames(session_groups)
        label_sessions = {
            label: [f for ts_frames in ts_dict.values() for f in ts_frames]
            for label, ts_dict in session_groups.items()
        }
        print(f"[Balance] Setiap label = {min_sessions} sesi")
    else:
        label_sessions = label_frames
        min_sessions   = None
else:
    label_sessions = scan_dataset(DATA_DIR)
    session_groups = None

    if BALANCE_DATASET:
        label_sessions, min_sessions = balance_label_sessions(label_sessions)
        print(f"[Balance] Setiap label = {min_sessions} sesi")
    else:
        min_sessions = None

n_labels   = len(label_sessions)
n_total    = sum(len(v) for v in label_sessions.values())
print(f"\nSubjek     : {n_labels}")
print(f"Total sesi : {n_total}")
print("\nDistribusi per subjek:")
for label, sessions in sorted(label_sessions.items()):
    print(f"  {label:<15} : {len(sessions)} {'frame' if frame_layout else 'sesi'}")

In [ ]:
# Cari normalizer (otomatis dari checkpoint pertama jika NORMALIZER_PATH=None)
normalizer = None
norm_path  = NORMALIZER_PATH

if norm_path is None and CHECKPOINTS:
    first_ckpt = next(iter(CHECKPOINTS.values()))
    if first_ckpt and Path(first_ckpt).exists():
        candidate = Path(first_ckpt).parent / "normalizer.json"
        if candidate.exists():
            norm_path = str(candidate)

if norm_path and Path(norm_path).exists():
    normalizer = GeometryNormalizer.load(norm_path)
    print(f"Normalizer  : {norm_path}")
else:
    print("Normalizer  : tidak digunakan (tidak ditemukan)")

# Buat dataset & loader
dataset = PalmPairDataset(
    label_sessions=label_sessions,
    n_points=N_POINTS,
    sampling=SAMPLING,
    augment=None,
    normalizer=normalizer,
)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)
print(f"\nTotal pasangan: {len(dataset)}")
genuine_count  = sum(1 for _, _, lbl in dataset.pairs if lbl == 1.0)
impostor_count = sum(1 for _, _, lbl in dataset.pairs if lbl == 0.0)
print(f"  Genuine  : {genuine_count}")
print(f"  Impostor : {impostor_count}")

## 4. Evaluasi Model

In [ ]:
all_results = []
roc_data    = {}
det_data    = {}

for model_name, ckpt_path in CHECKPOINTS.items():
    print(f"\n{'='*60}")
    print(f"Evaluasi : {model_name}")
    print(f"Checkpoint: {ckpt_path}")
    print(f"{'='*60}")

    if ckpt_path and Path(ckpt_path).exists():
        model = load_model(ckpt_path, GEOMETRY_DIM, DEVICE)
        print("Checkpoint berhasil dimuat.")
    else:
        print(f"[WARN] Checkpoint tidak ditemukan: {ckpt_path}")
        print("       Gunakan bobot acak (hasil tidak bermakna).")
        model = SiamesePalmNet(geom_dim=GEOMETRY_DIM).to(DEVICE)
        model.eval()

    result = evaluate_model(
        model=model,
        loader=loader,
        device=DEVICE,
        model_name=model_name,
        output_dir=OUTPUT_DIR,
        save_scores=SAVE_SCORES,
    )
    all_results.append(result)
    roc_data[model_name] = (result["labels"], result["scores"])
    det_data[model_name] = (result["labels"], result["scores"])

    print(f"\n  EER            = {result['eer']:.4f}")
    print(f"  AUC            = {result['auc']:.4f}")
    print(f"  TAR@FAR=1%     = {result['tar_at_far1']:.4f}")
    print(f"  TAR@FAR=0.1%   = {result['tar_at_far01']:.4f}")
    print(f"  d-prime        = {result['dprime']:.3f}")
    print(f"  Accuracy@EER   = {result['accuracy_at_eer']:.4f}")

print("\n" + "="*60)
print("Evaluasi selesai untuk semua model.")

## 5. Tabel Metrik Verifikasi

In [ ]:
print("\n" + "="*80)
print("EVALUATION RESULTS — VERIFIKASI BIOMETRIK")
print("="*80)
print_metrics_table(all_results)

# Simpan hasil ke JSON
summary_data = []
for r in all_results:
    summary_data.append({k: v for k, v in r.items()
                         if k not in ("labels", "scores", "embeddings")})

results_json_path = OUTPUT_DIR / "results.json"
with open(results_json_path, "w") as f:
    json.dump(summary_data, f, indent=2)

print(f"\nHasil tersimpan di: {results_json_path}")

## 6. Plot Kurva Evaluasi

In [ ]:
# ROC Curve
print("ROC Curve:")
plot_roc(roc_data, save_path=OUTPUT_DIR / "roc_curves.png")

In [ ]:
# DET Curve (Detection Error Tradeoff — FAR vs FRR)
print("DET Curve:")
plot_det(det_data, save_path=OUTPUT_DIR / "det_curves.png")

In [ ]:
# Similarity Distribution per model
for r in all_results:
    print(f"\nSimilarity Distribution — {r['model']}:")
    plot_similarity_dist(
        r["labels"], r["scores"],
        title=f"{r['model']} — Genuine vs Impostor",
        save_path=OUTPUT_DIR / f"{r['model']}_sim_dist.png",
    )

## 7. Identifikasi — Rank-1 & CMC Curve

Evaluasi **identifikasi tertutup** (closed-set): setiap probe dicocokkan ke
semua sesi gallery lainnya, kemudian diranking berdasarkan similarity tertinggi.

In [ ]:
def build_gallery_and_probes(label_sessions: dict, seed: int = 42) -> tuple:
    """
    Buat gallery (1 sesi per label) dan probe (sisa sesi).
    Gallery = sesi pertama per label, probe = sisanya.

    Returns:
        gallery_dirs   : {label: Path}
        probe_dirs     : [(label, Path)]
    """
    import random
    rng = random.Random(seed)
    gallery_dirs = {}
    probe_dirs   = []

    for label, sessions in label_sessions.items():
        shuffled = sessions.copy()
        rng.shuffle(shuffled)
        gallery_dirs[label] = shuffled[0]
        for s in shuffled[1:]:
            probe_dirs.append((label, s))

    return gallery_dirs, probe_dirs


@torch.no_grad()
def get_embedding(model, session_dir: Path, device,
                  n_points: int = 4096, normalizer=None) -> np.ndarray:
    """Dapatkan embedding 128-dim dari satu sesi."""
    from utils.dataset import load_session, _sample_points
    cloud, geom = load_session(session_dir)
    pts  = _sample_points(cloud, n_points, method="random")
    pts  = torch.from_numpy(pts).unsqueeze(0).to(device)   # (1, N, 6)
    geom_arr = geom.copy()
    if normalizer is not None:
        geom_arr = normalizer.transform(geom_arr)
    geom_t = torch.from_numpy(geom_arr).unsqueeze(0).to(device)  # (1, 17)

    # Forward satu branch: gunakan encoder langsung
    emb = model.encoder(pts, geom_t)  # (1, 128)
    return emb.squeeze(0).cpu().numpy()


def compute_cmc(rank_positions: list[int], max_rank: int = 10) -> np.ndarray:
    """CMC curve: kumulative recognition rate per rank."""
    cmc = np.zeros(max_rank)
    for pos in rank_positions:
        if pos <= max_rank:
            cmc[pos - 1:] += 1
    return cmc / len(rank_positions)


print("Fungsi identifikasi siap.")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine_sim

cmc_results = {}  # {model_name: cmc_curve}

for r in all_results:
    model_name = r["model"]
    ckpt_path  = CHECKPOINTS.get(model_name)

    print(f"\n{'='*60}")
    print(f"Identifikasi : {model_name}")

    if ckpt_path and Path(ckpt_path).exists():
        model_id = load_model(ckpt_path, GEOMETRY_DIM, DEVICE)
    else:
        model_id = SiamesePalmNet(geom_dim=GEOMETRY_DIM).to(DEVICE)
        model_id.eval()

    gallery_dirs, probe_dirs = build_gallery_and_probes(label_sessions)
    gallery_labels = list(gallery_dirs.keys())

    print(f"Gallery : {len(gallery_labels)} subjek")
    print(f"Probe   : {len(probe_dirs)} sesi")

    # Komputasi embedding gallery
    gallery_embs = np.stack([
        get_embedding(model_id, gallery_dirs[lbl], DEVICE, N_POINTS, normalizer)
        for lbl in gallery_labels
    ])  # (n_labels, 128)

    # Komputasi embedding probe dan hitung rank
    rank_positions   = []
    correct_rank1    = 0
    total_probes     = len(probe_dirs)
    probe_true_labels   = []
    probe_pred_labels   = []

    for true_label, probe_dir in probe_dirs:
        probe_emb = get_embedding(model_id, probe_dir, DEVICE, N_POINTS, normalizer)
        sims      = sk_cosine_sim(probe_emb.reshape(1, -1), gallery_embs).flatten()
        ranking   = np.argsort(sims)[::-1]  # descending
        ranked_labels = [gallery_labels[i] for i in ranking]

        pred_rank1 = ranked_labels[0]
        probe_true_labels.append(true_label)
        probe_pred_labels.append(pred_rank1)

        # Posisi rank dari label benar
        rank_pos = ranked_labels.index(true_label) + 1
        rank_positions.append(rank_pos)
        if pred_rank1 == true_label:
            correct_rank1 += 1

    rank1   = correct_rank1 / total_probes
    rank5   = sum(1 for p in rank_positions if p <= 5)  / total_probes
    rank10  = sum(1 for p in rank_positions if p <= 10) / total_probes
    cmc_curve = compute_cmc(rank_positions, max_rank=len(gallery_labels))
    cmc_results[model_name] = {
        "cmc": cmc_curve,
        "rank_positions": rank_positions,
        "true_labels": probe_true_labels,
        "pred_labels": probe_pred_labels,
    }

    print(f"\nRank-1  : {rank1:.4f} ({correct_rank1}/{total_probes})")
    print(f"Rank-5  : {rank5:.4f}")
    print(f"Rank-10 : {rank10:.4f}")

print("\nIdentifikasi selesai.")

In [ ]:
# CMC Curve
fig, ax = plt.subplots(figsize=(7, 5))

for model_name, cmc_info in cmc_results.items():
    cmc   = cmc_info["cmc"]
    ranks = np.arange(1, len(cmc) + 1)
    ax.plot(ranks, cmc * 100, marker="o", markersize=4, lw=2,
            label=f"{model_name}  (Rank-1={cmc[0]*100:.1f}%)")

ax.set_xlabel("Rank")
ax.set_ylabel("Cumulative Recognition Rate (%)")
ax.set_title("CMC Curve — Palm Identification")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 105)
ax.set_xlim(1, max(len(v["cmc"]) for v in cmc_results.values()))
fig.tight_layout()
cmc_path = OUTPUT_DIR / "cmc_curve.png"
fig.savefig(cmc_path, dpi=150)
print(f"CMC curve disimpan: {cmc_path}")
plt.show()

## 8. Confusion Matrix (Identifikasi Rank-1)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

for model_name, cmc_info in cmc_results.items():
    true_labels = cmc_info["true_labels"]
    pred_labels = cmc_info["pred_labels"]

    all_labels_sorted = sorted(set(true_labels))
    cm = confusion_matrix(true_labels, pred_labels, labels=all_labels_sorted)

    n_subj = len(all_labels_sorted)
    fig_size = max(8, n_subj * 0.8)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size * 0.9))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=all_labels_sorted)
    disp.plot(ax=ax, colorbar=False, xticks_rotation=45, cmap="Blues")
    ax.set_title(f"{model_name} — Confusion Matrix (Rank-1 Identification)")
    fig.tight_layout()
    cm_path = OUTPUT_DIR / f"{model_name}_confusion_matrix.png"
    fig.savefig(cm_path, dpi=150, bbox_inches="tight")
    print(f"Confusion matrix disimpan: {cm_path}")
    plt.show()

## 9. Similarity Score per Subjek

Visualisasi rata-rata similarity genuine dan impostor per subjek.
Berguna untuk mengidentifikasi subjek yang sulit dibedakan.

In [ ]:
def compute_per_subject_similarity(pairs, scores, label_sessions):
    """
    Hitung rata-rata similarity genuine dan impostor per subjek.

    Args:
        pairs         : list of (dir_a, dir_b, label)
        scores        : np.ndarray (N,)
        label_sessions: {label: [Path]}

    Returns:
        dict: {subject: {"genuine_mean": float, "impostor_mean": float, "n_genuine": int, "n_impostor": int}}
    """
    # Buat mapping dir → label
    dir_to_label = {}
    for label, sessions in label_sessions.items():
        for s in sessions:
            dir_to_label[s] = label

    per_subject = {label: {"genuine": [], "impostor": []}
                   for label in label_sessions}

    for (dir_a, dir_b, lbl), score in zip(pairs, scores):
        label_a = dir_to_label.get(dir_a)
        label_b = dir_to_label.get(dir_b)
        if label_a is None or label_b is None:
            continue
        if lbl == 1.0:  # genuine
            per_subject[label_a]["genuine"].append(score)
        else:           # impostor
            per_subject[label_a]["impostor"].append(score)
            per_subject[label_b]["impostor"].append(score)

    result = {}
    for subj, data in per_subject.items():
        gen  = data["genuine"]
        imp  = data["impostor"]
        result[subj] = {
            "genuine_mean":  float(np.mean(gen))  if gen  else 0.0,
            "impostor_mean": float(np.mean(imp)) if imp else 0.0,
            "n_genuine":  len(gen),
            "n_impostor": len(imp),
        }
    return result

print("Fungsi per-subject similarity siap.")

In [ ]:
for r in all_results:
    model_name = r["model"]
    pairs  = dataset.pairs
    scores = r["scores"]

    per_subj = compute_per_subject_similarity(pairs, scores, label_sessions)
    subjects = sorted(per_subj.keys())

    gen_means  = [per_subj[s]["genuine_mean"]  for s in subjects]
    imp_means  = [per_subj[s]["impostor_mean"] for s in subjects]

    x     = np.arange(len(subjects))
    width = 0.35

    fig, ax = plt.subplots(figsize=(max(8, len(subjects) * 0.7), 5))
    ax.bar(x - width/2, gen_means, width, label="Genuine",  color="steelblue", alpha=0.8)
    ax.bar(x + width/2, imp_means, width, label="Impostor", color="salmon",    alpha=0.8)

    # Garis EER threshold
    eer_thresh = r["eer_threshold"]
    ax.axhline(eer_thresh, color="black", linestyle="--", lw=1.2,
               label=f"EER threshold = {eer_thresh:.3f}")

    ax.set_xticks(x)
    ax.set_xticklabels(subjects, rotation=45, ha="right")
    ax.set_xlabel("Subjek")
    ax.set_ylabel("Rata-rata Cosine Similarity")
    ax.set_title(f"{model_name} — Similarity Score per Subjek")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    sim_path = OUTPUT_DIR / f"{model_name}_per_subject_sim.png"
    fig.savefig(sim_path, dpi=150)
    print(f"Per-subject similarity disimpan: {sim_path}")
    plt.show()

    # Cetak tabel ringkas
    print(f"\n{'Subjek':<15} {'Gen Mean':>9} {'Imp Mean':>9} {'Separasi':>9} {'n_gen':>6}")
    print("-" * 55)
    for s in subjects:
        g = per_subj[s]["genuine_mean"]
        i = per_subj[s]["impostor_mean"]
        ng = per_subj[s]["n_genuine"]
        print(f"{s:<15} {g:>9.4f} {i:>9.4f} {g-i:>9.4f} {ng:>6}")

## 10. t-SNE Embedding Space

In [ ]:
# t-SNE untuk model terakhir
# Gunakan embs_a (satu embedding per pasangan agar tidak ada duplikat berlebihan)
if all_results:
    last = all_results[-1]

    # Buat integer label per embedding dari pairs
    all_dirs  = [pair[0] for pair in dataset.pairs]
    dir_to_label = {}
    label_to_int = {}
    int_counter  = 0
    for label, sessions in label_sessions.items():
        if label not in label_to_int:
            label_to_int[label] = int_counter
            int_counter += 1
        for s in sessions:
            dir_to_label[s] = label_to_int[label]

    emb_labels = np.array([
        dir_to_label.get(pair[0], 0) for pair in dataset.pairs
    ])

    print(f"t-SNE untuk {last['model']} — {len(last['embeddings'])} embeddings...")
    plot_tsne(
        last["embeddings"],
        emb_labels,
        title=f"{last['model']} — Embedding Space (t-SNE)",
        save_path=OUTPUT_DIR / f"{last['model']}_tsne.png",
    )

## 11. Ablation Study — Ringkasan Multi-Model

Bandingkan semua model yang dievaluasi dalam satu tabel.

In [ ]:
if len(all_results) > 1:
    print("\n" + "="*80)
    print("ABLATION STUDY — Perbandingan Semua Model")
    print("="*80)
    print_metrics_table(all_results)

    # Bar chart EER
    model_names = [r["model"] for r in all_results]
    eers        = [r["eer"]   for r in all_results]
    aucs        = [r["auc"]   for r in all_results]
    dprimes     = [r["dprime"] for r in all_results]
    tar1s       = [r["tar_at_far1"] for r in all_results]

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    for ax, vals, title, lower_better in zip(
        axes,
        [eers, aucs, dprimes, tar1s],
        ["EER (↓)", "AUC (↑)", "d-prime (↑)", "TAR@FAR=1% (↑)"],
        [True, False, False, False],
    ):
        colors = ["steelblue"] * len(vals)
        if lower_better:
            colors[np.argmin(vals)] = "green"
        else:
            colors[np.argmax(vals)] = "green"

        bars = ax.bar(model_names, vals, color=colors, alpha=0.8, edgecolor="black")
        ax.set_title(title)
        ax.set_ylabel(title.split(" ")[0])
        ax.set_xticklabels(model_names, rotation=30, ha="right")
        ax.grid(axis="y", alpha=0.3)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=9)

    fig.suptitle("Ablation Study — Perbandingan Metrik Model", y=1.02)
    fig.tight_layout()
    ablation_path = OUTPUT_DIR / "ablation_comparison.png"
    fig.savefig(ablation_path, dpi=150, bbox_inches="tight")
    print(f"\nAblation chart disimpan: {ablation_path}")
    plt.show()
else:
    print("Hanya 1 model — tidak ada ablation study.")
    print("Tambahkan lebih banyak checkpoint di bagian KONFIGURASI (sel 2).")

## 12. Ringkasan Output

Semua file hasil evaluasi tersimpan di folder `eval_results/`.

In [ ]:
print("=" * 60)
print(f"EVALUASI SELESAI")
print("=" * 60)
print(f"Output folder: {OUTPUT_DIR.resolve()}")
print()
print("File yang dihasilkan:")
for f in sorted(OUTPUT_DIR.glob("*")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45} {size_kb:>7.1f} KB")

print()
print("Metrik terbaik:")
if all_results:
    best_eer_model = min(all_results, key=lambda r: r["eer"])
    best_auc_model = max(all_results, key=lambda r: r["auc"])
    print(f"  EER terkecil    : {best_eer_model['model']}  EER={best_eer_model['eer']:.4f}")
    print(f"  AUC terbesar    : {best_auc_model['model']}  AUC={best_auc_model['auc']:.4f}")

if cmc_results:
    print("  Rank-1 per model:")
    for model_name, cmc_info in cmc_results.items():
        r1 = cmc_info["cmc"][0]
        print(f"    {model_name:<20} Rank-1 = {r1*100:.1f}%")